In [5]:
import pandas as pd
import numpy as np

LAMBDA = -11.72
EPS = 1e-12

def load_probs(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    df["has_pairs"] = df["total_found"].fillna(0) > 0
    df["score"] = df["avg_log_prob"].astype(float)
    df.loc[~df["has_pairs"], "score"] = np.nan
    df["label"] = np.where(
        df["score"].isna(), "no_edges",
        np.where(df["score"] < LAMBDA, "joke", "literary")
    )
    return df

um = load_probs("um_sentences_probabilities_2.csv")
lit = load_probs("lit_sentences_probabilities_2.csv")

def summarize(df: pd.DataFrame, name: str):
    n = len(df)
    vc = df["label"].value_counts().reindex(["joke", "literary", "no_edges"], fill_value=0)
    print(f"\n{name}")
    for lab in ["joke", "literary", "no_edges"]:
        print(f"{lab:9s}: {vc[lab]:4d} ({vc[lab]/n*100:5.1f}%)")
    # отдельно: среди строк, где пары вообще есть
    sub = df[df["has_pairs"]]
    if len(sub):
        share_joke = (sub["label"] == "joke").mean() * 100
        print(f"среди has_pairs (n={len(sub)}): joke = {share_joke:.1f}%, "
              f"literary = {100-share_joke:.1f}%")

summarize(um, "юмор")
summarize(lit, "литература")

# сводная матрица
summary = pd.DataFrame({
    "um":  um["label"].value_counts().reindex(["joke","literary","no_edges"], fill_value=0),
    "lit": lit["label"].value_counts().reindex(["joke","literary","no_edges"], fill_value=0),
})
summary.loc["ИТОГО"] = summary.sum()
print("\nСводная таблица:\n", summary)


юмор
joke     :    0 (  0.0%)
literary :    9 ( 12.2%)
no_edges :   65 ( 87.8%)
среди has_pairs (n=9): joke = 0.0%, literary = 100.0%

литература
joke     :    0 (  0.0%)
literary :   24 ( 32.4%)
no_edges :   50 ( 67.6%)
среди has_pairs (n=24): joke = 0.0%, literary = 100.0%

Сводная таблица:
           um  lit
label            
joke       0    0
literary   9   24
no_edges  65   50
ИТОГО     74   74


In [7]:
for name, df in [("um", um), ("lit", lit)]:
    print(f"\n{name}: доля joke в зависимости от found_ratio")
    sub = df[df["has_pairs"]].copy()
    sub["found_ratio_bin"] = pd.cut(sub["found_ratio"], bins=[-0.01, 0.0, 0.5, 0.99, 1.0],
                                    labels=["0", "0-0.5", "0.5-1", "1"])
    print(sub.groupby("found_ratio_bin", observed=True)["label"]
              .value_counts(normalize=True).unstack(fill_value=0).round(2))


um: доля joke в зависимости от found_ratio
label            literary
found_ratio_bin          
1                     1.0

lit: доля joke в зависимости от found_ratio
label            literary
found_ratio_bin          
0-0.5                 1.0
1                     1.0
